# 🔄 CDC Snapshotting — Log Compaction with Trino & Iceberg

Turn an **append-only stream of database changes** (CDC logs) into a highly efficient, materialized **"Snapshot"** of the current state — without Kafka or Flink.

This notebook demonstrates the **Snapshotting / Log Compaction** pattern using batch processing with **Trino** and **Iceberg's `MERGE INTO`** capabilities.

---

### What is CDC?

**Change Data Capture (CDC)** records every row-level change (`INSERT`, `UPDATE`, `DELETE`) from a source database as an immutable event in a log. Tools like **Debezium**, **AWS DMS**, or **Fivetran** produce these logs.

### Why Snapshotting?

CDC logs grow forever. To answer _"What does the current state of the users table look like?"_, you need to **compact** (or **fold**) the log into the latest state for each primary key — exactly like Kafka log compaction, but done in batch SQL.

### The Pinterest Parallel

This pattern mirrors [Pinterest's Next-Generation DB Ingestion](https://medium.com/pinterest-engineering/next-generation-db-ingestion-at-pinterest-66844b7153b7), where CDC events are landed into an append-only Iceberg table and periodically merged into a snapshot table.

---

### Architecture

```
Source DB ──CDC──▶ 🥉 Bronze (append-only ledger) ──MERGE──▶ 🥈 Silver (current-state snapshot)
                  Every I/U/D event logged             Dedup + MERGE INTO for latest state
```

**Prerequisites:** Run `setup.ipynb` first to create the lakehouse infrastructure (S3 bucket, Polaris catalog, Trino connection).

---
## ⚙️ Connect to Trino

In [1]:
from trino.dbapi import connect

TRINO_HOST = "trino"
TRINO_PORT = 8080

conn = connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user="admin",
    catalog="iceberg",
    schema="bronze",
)
cursor = conn.cursor()


def run_query(sql, display=True):
    """Execute a query and return results as a formatted table."""
    cursor.execute(sql)
    try:
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        if display and rows:
            widths = [
                max(len(str(c)), max(len(str(r[i])) for r in rows))
                for i, c in enumerate(columns)
            ]
            header = " | ".join(c.ljust(w) for c, w in zip(columns, widths))
            sep = "-+-".join("-" * w for w in widths)
            print(header)
            print(sep)
            for row in rows:
                print(" | ".join(str(v).ljust(w) for v, w in zip(row, widths)))
        return rows
    except Exception:
        return []


print("✅ Connected to Trino")

✅ Connected to Trino


---
## 🥉 The Bronze Layer: Append-Only CDC Ledger

This table mimics what a tool like **Debezium** or **AWS DMS** would output. It records **every single database operation** as an immutable event:

| Column | Description |
|--------|-------------|
| `event_id` | Unique identifier for this CDC event |
| `op` | Operation type: **I** (Insert), **U** (Update), **D** (Delete) |
| `user_id` | Primary key of the source `users` table |
| `name` | User's current name at time of event |
| `email` | User's current email at time of event |
| `city` | User's current city at time of event |
| `event_ts` | When the change occurred in the source DB |

**Key insight:** This table is **append-only**. We never update or delete rows here — every change is a new row.

In [3]:
run_query("""
CREATE TABLE IF NOT EXISTS iceberg.bronze.cdc_users_log (
    event_id   VARCHAR,
    op         VARCHAR,
    user_id    BIGINT,
    name       VARCHAR,
    email      VARCHAR,
    city       VARCHAR,
    event_ts   TIMESTAMP(6) WITH TIME ZONE
) WITH (format = 'PARQUET')
""")
print("✅ Table 'cdc_users_log' created in bronze layer")

✅ Table 'cdc_users_log' created in bronze layer


### 📥 Simulate CDC Events — Batch 1 (Initial Inserts)

The first batch of CDC events represents new users being created in the source database:

| # | Operation | Description |
|---|-----------|-------------|
| 1 | `I` (Insert) | Alice joins from Riyadh |
| 2 | `I` (Insert) | Bob joins from Jeddah |
| 3 | `I` (Insert) | Charlie joins from Dammam |

In [4]:
run_query("""
INSERT INTO iceberg.bronze.cdc_users_log VALUES
    ('evt001', 'I', 1, 'Alice',   'alice@old.com',    'Riyadh', TIMESTAMP '2026-02-24 08:00:00.000000 UTC'),
    ('evt002', 'I', 2, 'Bob',     'bob@email.com',    'Jeddah', TIMESTAMP '2026-02-24 08:05:00.000000 UTC'),
    ('evt003', 'I', 3, 'Charlie', 'charlie@email.com', 'Dammam', TIMESTAMP '2026-02-24 08:10:00.000000 UTC')
""")
print("✅ 3 CDC events ingested (Batch 1 — initial inserts)")

rows
----
3   
✅ 3 CDC events ingested (Batch 1 — initial inserts)


---
## 🥈 The Silver Layer: Current State Snapshot

This table represents the **actual, current state** of the `users` table in the source database. While CDC gives us every change, the snapshot gives us the answer to: _"What does the table look like **right now**?"_

In [5]:
run_query("""
CREATE TABLE IF NOT EXISTS iceberg.silver.users_snapshot (
    user_id    BIGINT,
    name       VARCHAR,
    email      VARCHAR,
    city       VARCHAR,
    updated_at TIMESTAMP(6) WITH TIME ZONE
) WITH (format = 'PARQUET')
""")
print("✅ Table 'users_snapshot' created in silver layer")

✅ Table 'users_snapshot' created in silver layer


### 🔄 Initial Sync — MERGE Batch 1 into Snapshot

Since Batch 1 has no duplicate `user_id` values, a simple MERGE works perfectly to establish the initial snapshot.

In [6]:
run_query("""
MERGE INTO iceberg.silver.users_snapshot AS tgt
USING iceberg.bronze.cdc_users_log AS src
ON tgt.user_id = src.user_id
WHEN MATCHED THEN
    UPDATE SET name = src.name, email = src.email, city = src.city, updated_at = src.event_ts
WHEN NOT MATCHED THEN
    INSERT (user_id, name, email, city, updated_at)
    VALUES (src.user_id, src.name, src.email, src.city, src.event_ts)
""")
print("✅ Initial sync complete — 3 users in snapshot")

rows
----
3   
✅ Initial sync complete — 3 users in snapshot


In [7]:
print("📋 Current snapshot (baseline):")
print()
run_query("SELECT * FROM iceberg.silver.users_snapshot ORDER BY user_id")

📋 Current snapshot (baseline):

user_id | name    | email             | city   | updated_at               
--------+---------+-------------------+--------+--------------------------
1       | Alice   | alice@old.com     | Riyadh | 2026-02-24 08:00:00+00:00
2       | Bob     | bob@email.com     | Jeddah | 2026-02-24 08:05:00+00:00
3       | Charlie | charlie@email.com | Dammam | 2026-02-24 08:10:00+00:00


[[1,
  'Alice',
  'alice@old.com',
  'Riyadh',
  datetime.datetime(2026, 2, 24, 8, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 [2,
  'Bob',
  'bob@email.com',
  'Jeddah',
  datetime.datetime(2026, 2, 24, 8, 5, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 [3,
  'Charlie',
  'charlie@email.com',
  'Dammam',
  datetime.datetime(2026, 2, 24, 8, 10, tzinfo=zoneinfo.ZoneInfo(key='UTC'))]]

---
### 📥 Simulate CDC Events — Batch 2 (Updates & Deletes)

Now new changes arrive from the source database:

| # | Operation | Description |
|---|-----------|-------------|
| 4 | `U` (Update) | Alice moves to Dubai |
| 5 | `U` (Update) | Alice updates her email |
| 6 | `U` (Update) | Bob moves to Makkah |
| 7 | `D` (Delete) | Charlie's account is deleted |

Notice that **Alice has 2 updates** in this batch, **plus** her original insert is still in the log — she now has **3 total events** across the CDC log. This is the key challenge!

In [8]:
run_query("""
INSERT INTO iceberg.bronze.cdc_users_log VALUES
    ('evt004', 'U', 1, 'Alice',   'alice@old.com',    'Dubai',  TIMESTAMP '2026-02-24 09:00:00.000000 UTC'),
    ('evt005', 'U', 1, 'Alice',   'alice@new.com',    'Dubai',  TIMESTAMP '2026-02-24 10:30:00.000000 UTC'),
    ('evt006', 'U', 2, 'Bob',     'bob@email.com',    'Makkah', TIMESTAMP '2026-02-24 11:00:00.000000 UTC'),
    ('evt007', 'D', 3, 'Charlie', 'charlie@email.com', 'Dammam', TIMESTAMP '2026-02-24 12:00:00.000000 UTC')
""")
print("✅ 4 CDC events ingested (Batch 2 — updates & deletes)")

rows
----
4   
✅ 4 CDC events ingested (Batch 2 — updates & deletes)


### 🔍 View the Full CDC Log

The Bronze CDC log now has **7 events** — and `user_id = 1` (Alice) appears **3 times**.

In [9]:
run_query("SELECT event_id, op, user_id, name, email, city, event_ts FROM iceberg.bronze.cdc_users_log ORDER BY event_ts")

event_id | op | user_id | name    | email             | city   | event_ts                 
---------+----+---------+---------+-------------------+--------+--------------------------
evt001   | I  | 1       | Alice   | alice@old.com     | Riyadh | 2026-02-24 08:00:00+00:00
evt002   | I  | 2       | Bob     | bob@email.com     | Jeddah | 2026-02-24 08:05:00+00:00
evt003   | I  | 3       | Charlie | charlie@email.com | Dammam | 2026-02-24 08:10:00+00:00
evt004   | U  | 1       | Alice   | alice@old.com     | Dubai  | 2026-02-24 09:00:00+00:00
evt005   | U  | 1       | Alice   | alice@new.com     | Dubai  | 2026-02-24 10:30:00+00:00
evt006   | U  | 2       | Bob     | bob@email.com     | Makkah | 2026-02-24 11:00:00+00:00
evt007   | D  | 3       | Charlie | charlie@email.com | Dammam | 2026-02-24 12:00:00+00:00


[['evt001',
  'I',
  1,
  'Alice',
  'alice@old.com',
  'Riyadh',
  datetime.datetime(2026, 2, 24, 8, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt002',
  'I',
  2,
  'Bob',
  'bob@email.com',
  'Jeddah',
  datetime.datetime(2026, 2, 24, 8, 5, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt003',
  'I',
  3,
  'Charlie',
  'charlie@email.com',
  'Dammam',
  datetime.datetime(2026, 2, 24, 8, 10, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt004',
  'U',
  1,
  'Alice',
  'alice@old.com',
  'Dubai',
  datetime.datetime(2026, 2, 24, 9, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt005',
  'U',
  1,
  'Alice',
  'alice@new.com',
  'Dubai',
  datetime.datetime(2026, 2, 24, 10, 30, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt006',
  'U',
  2,
  'Bob',
  'bob@email.com',
  'Makkah',
  datetime.datetime(2026, 2, 24, 11, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt007',
  'D',
  3,
  'Charlie',
  'charlie@email.com',
  'Dammam',
  datetime.datetime(2026, 2, 24, 12, 0, tzinfo=zoneinfo.Zone

---
### ❌ The Naïve Approach — Why It Fails

Your first instinct might be to `MERGE INTO` the snapshot directly from the full CDC log:

```sql
MERGE INTO silver.users_snapshot AS tgt
USING  bronze.cdc_users_log   AS src
ON     tgt.user_id = src.user_id
WHEN MATCHED THEN UPDATE ...
WHEN NOT MATCHED THEN INSERT ...
```

**This will FAIL!** ❌

Alice (`user_id = 1`) **already exists** in the snapshot from Batch 1, and she has **3 events** in the CDC log. Trino's `MERGE` requires that each **existing target row** matches **at most one source row**. Since Alice in the target matches 3 Alice rows in the source, this causes an error.

Let's prove it:

In [10]:
try:
    run_query("""
    MERGE INTO iceberg.silver.users_snapshot AS tgt
    USING iceberg.bronze.cdc_users_log AS src
    ON tgt.user_id = src.user_id
    WHEN MATCHED THEN
        UPDATE SET name = src.name, email = src.email, city = src.city, updated_at = src.event_ts
    WHEN NOT MATCHED THEN
        INSERT (user_id, name, email, city, updated_at)
        VALUES (src.user_id, src.name, src.email, src.city, src.event_ts)
    """)
    print("Merge succeeded (unexpected!)")
except Exception as e:
    print(f"❌ MERGE FAILED (as expected!)")
    print(f"\nError: {e}")
    print(f"\n💡 Reason: user_id=1 (Alice) exists in the snapshot and has 3 events in the CDC log.")
    print(f"   Trino requires each target row to match at most ONE source row.")
    print(f"   Multiple source rows matching the same target row causes a conflict.")

❌ MERGE FAILED (as expected!)

Error: TrinoUserError(type=USER_ERROR, name=MERGE_TARGET_ROW_MULTIPLE_MATCHES, message="One MERGE target table row matched more than one source row", query_id=20260225_091707_00008_k4ixh)

💡 Reason: user_id=1 (Alice) exists in the snapshot and has 3 events in the CDC log.
   Trino requires each target row to match at most ONE source row.
   Multiple source rows matching the same target row causes a conflict.


---
### ✅ The Correct Approach — CTE + Window Function Deduplication

To fix this, we use a **Common Table Expression (CTE)** with a **Window Function** to isolate only the **most recent event per `user_id`** before applying the `MERGE`.

```
CDC Log (7 events)                  Dedup CTE (3 events)             Snapshot (2 rows)
┌─────────────────────┐            ┌─────────────────────┐          ┌─────────────────────┐
│ evt001 I Alice  RUH │            │                     │          │                     │
│ evt004 U Alice  DXB │ ──dedup──▶ │ evt005 U Alice  DXB │ ─merge─▶ │ Alice  DXB (upsert) │
│ evt005 U Alice  DXB │            │ evt006 U Bob    MKX │          │ Bob    MKX (upsert) │
│ evt002 I Bob    JED │            │ evt007 D Charlie DMM │          │ (Charlie deleted)   │
│ evt006 U Bob    MKX │            └─────────────────────┘          └─────────────────────┘
│ evt003 I Charlie DMM│
│ evt007 D Charlie DMM│
└─────────────────────┘
```

The window function `ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts DESC)` ranks events per user by recency. We keep only `rn = 1` — the latest event for each user.

In [11]:
# First, let's see what the dedup CTE produces
print("📋 Deduplicated CDC events (latest per user_id):")
print()
run_query("""
WITH deduped AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY event_ts DESC
        ) AS rn
    FROM iceberg.bronze.cdc_users_log
)
SELECT event_id, op, user_id, name, email, city, event_ts
FROM deduped
WHERE rn = 1
ORDER BY user_id
""")

📋 Deduplicated CDC events (latest per user_id):

event_id | op | user_id | name    | email             | city   | event_ts                 
---------+----+---------+---------+-------------------+--------+--------------------------
evt005   | U  | 1       | Alice   | alice@new.com     | Dubai  | 2026-02-24 10:30:00+00:00
evt006   | U  | 2       | Bob     | bob@email.com     | Makkah | 2026-02-24 11:00:00+00:00
evt007   | D  | 3       | Charlie | charlie@email.com | Dammam | 2026-02-24 12:00:00+00:00


[['evt005',
  'U',
  1,
  'Alice',
  'alice@new.com',
  'Dubai',
  datetime.datetime(2026, 2, 24, 10, 30, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt006',
  'U',
  2,
  'Bob',
  'bob@email.com',
  'Makkah',
  datetime.datetime(2026, 2, 24, 11, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt007',
  'D',
  3,
  'Charlie',
  'charlie@email.com',
  'Dammam',
  datetime.datetime(2026, 2, 24, 12, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))]]

In [12]:
# Now apply the full MERGE with deduplication
run_query("""
MERGE INTO iceberg.silver.users_snapshot AS tgt
USING (
    WITH deduped AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY event_ts DESC
            ) AS rn
        FROM iceberg.bronze.cdc_users_log
    )
    SELECT op, user_id, name, email, city, event_ts
    FROM deduped
    WHERE rn = 1
) AS src
ON tgt.user_id = src.user_id
WHEN MATCHED AND src.op = 'D' THEN
    DELETE
WHEN MATCHED AND src.op != 'D' THEN
    UPDATE SET
        name       = src.name,
        email      = src.email,
        city       = src.city,
        updated_at = src.event_ts
WHEN NOT MATCHED AND src.op != 'D' THEN
    INSERT (user_id, name, email, city, updated_at)
    VALUES (src.user_id, src.name, src.email, src.city, src.event_ts)
""")
print("✅ MERGE completed successfully — snapshot materialized!")

rows
----
3   
✅ MERGE completed successfully — snapshot materialized!


### 🔍 View the Current State Snapshot

The snapshot should show:
- **Alice** — in Dubai, with her new email ✅
- **Bob** — moved to Makkah ✅
- **Charlie** — deleted, should NOT appear ✅

In [13]:
run_query("SELECT * FROM iceberg.silver.users_snapshot ORDER BY user_id")

user_id | name  | email         | city   | updated_at               
--------+-------+---------------+--------+--------------------------
1       | Alice | alice@new.com | Dubai  | 2026-02-24 10:30:00+00:00
2       | Bob   | bob@email.com | Makkah | 2026-02-24 11:00:00+00:00


[[1,
  'Alice',
  'alice@new.com',
  'Dubai',
  datetime.datetime(2026, 2, 24, 10, 30, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 [2,
  'Bob',
  'bob@email.com',
  'Makkah',
  datetime.datetime(2026, 2, 24, 11, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))]]

---
## 🔄 Simulate a Third Batch

In production, CDC events arrive continuously. Let's simulate another batch of changes:

| # | Operation | Description |
|---|-----------|-------------|
| 8 | `I` (Insert) | Dana joins from Cairo |
| 9 | `U` (Update) | Alice moves to London |
| 10 | `U` (Update) | Bob updates his email |

In [14]:
run_query("""
INSERT INTO iceberg.bronze.cdc_users_log VALUES
    ('evt008', 'I', 4, 'Dana',  'dana@email.com',  'Cairo',  TIMESTAMP '2026-02-24 14:00:00.000000 UTC'),
    ('evt009', 'U', 1, 'Alice', 'alice@new.com',   'London', TIMESTAMP '2026-02-24 15:00:00.000000 UTC'),
    ('evt010', 'U', 2, 'Bob',   'bob@newmail.com', 'Makkah', TIMESTAMP '2026-02-24 16:00:00.000000 UTC')
""")
print("✅ 3 CDC events ingested (Batch 3)")

rows
----
3   
✅ 3 CDC events ingested (Batch 3)


### 📋 Full CDC Log (All Batches)

In [15]:
run_query("SELECT event_id, op, user_id, name, email, city, event_ts FROM iceberg.bronze.cdc_users_log ORDER BY event_ts")

event_id | op | user_id | name    | email             | city   | event_ts                 
---------+----+---------+---------+-------------------+--------+--------------------------
evt001   | I  | 1       | Alice   | alice@old.com     | Riyadh | 2026-02-24 08:00:00+00:00
evt002   | I  | 2       | Bob     | bob@email.com     | Jeddah | 2026-02-24 08:05:00+00:00
evt003   | I  | 3       | Charlie | charlie@email.com | Dammam | 2026-02-24 08:10:00+00:00
evt004   | U  | 1       | Alice   | alice@old.com     | Dubai  | 2026-02-24 09:00:00+00:00
evt005   | U  | 1       | Alice   | alice@new.com     | Dubai  | 2026-02-24 10:30:00+00:00
evt006   | U  | 2       | Bob     | bob@email.com     | Makkah | 2026-02-24 11:00:00+00:00
evt007   | D  | 3       | Charlie | charlie@email.com | Dammam | 2026-02-24 12:00:00+00:00
evt008   | I  | 4       | Dana    | dana@email.com    | Cairo  | 2026-02-24 14:00:00+00:00
evt009   | U  | 1       | Alice   | alice@new.com     | London | 2026-02-24 15:00:00+00:00

[['evt001',
  'I',
  1,
  'Alice',
  'alice@old.com',
  'Riyadh',
  datetime.datetime(2026, 2, 24, 8, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt002',
  'I',
  2,
  'Bob',
  'bob@email.com',
  'Jeddah',
  datetime.datetime(2026, 2, 24, 8, 5, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt003',
  'I',
  3,
  'Charlie',
  'charlie@email.com',
  'Dammam',
  datetime.datetime(2026, 2, 24, 8, 10, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt004',
  'U',
  1,
  'Alice',
  'alice@old.com',
  'Dubai',
  datetime.datetime(2026, 2, 24, 9, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt005',
  'U',
  1,
  'Alice',
  'alice@new.com',
  'Dubai',
  datetime.datetime(2026, 2, 24, 10, 30, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt006',
  'U',
  2,
  'Bob',
  'bob@email.com',
  'Makkah',
  datetime.datetime(2026, 2, 24, 11, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['evt007',
  'D',
  3,
  'Charlie',
  'charlie@email.com',
  'Dammam',
  datetime.datetime(2026, 2, 24, 12, 0, tzinfo=zoneinfo.Zone

### 🔄 Re-run the MERGE Pipeline

Same dedup+merge pattern — works idempotently regardless of how many batches have accumulated.

In [16]:
run_query("""
MERGE INTO iceberg.silver.users_snapshot AS tgt
USING (
    WITH deduped AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY event_ts DESC
            ) AS rn
        FROM iceberg.bronze.cdc_users_log
    )
    SELECT op, user_id, name, email, city, event_ts
    FROM deduped
    WHERE rn = 1
) AS src
ON tgt.user_id = src.user_id
WHEN MATCHED AND src.op = 'D' THEN
    DELETE
WHEN MATCHED AND src.op != 'D' THEN
    UPDATE SET
        name       = src.name,
        email      = src.email,
        city       = src.city,
        updated_at = src.event_ts
WHEN NOT MATCHED AND src.op != 'D' THEN
    INSERT (user_id, name, email, city, updated_at)
    VALUES (src.user_id, src.name, src.email, src.city, src.event_ts)
""")
print("✅ MERGE completed — snapshot updated with Batch 3!")

rows
----
3   
✅ MERGE completed — snapshot updated with Batch 3!


### 🔍 Updated Snapshot

Expected result:
- **Alice** — now in London ✅
- **Bob** — new email `bob@newmail.com` ✅
- **Dana** — new user from Cairo ✅
- **Charlie** — still deleted, should NOT appear ✅

In [17]:
run_query("SELECT * FROM iceberg.silver.users_snapshot ORDER BY user_id")

user_id | name  | email           | city   | updated_at               
--------+-------+-----------------+--------+--------------------------
1       | Alice | alice@new.com   | London | 2026-02-24 15:00:00+00:00
2       | Bob   | bob@newmail.com | Makkah | 2026-02-24 16:00:00+00:00
4       | Dana  | dana@email.com  | Cairo  | 2026-02-24 14:00:00+00:00


[[1,
  'Alice',
  'alice@new.com',
  'London',
  datetime.datetime(2026, 2, 24, 15, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 [2,
  'Bob',
  'bob@newmail.com',
  'Makkah',
  datetime.datetime(2026, 2, 24, 16, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 [4,
  'Dana',
  'dana@email.com',
  'Cairo',
  datetime.datetime(2026, 2, 24, 14, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC'))]]

---
## 🧊 Iceberg Time Travel

Every `MERGE` creates an **immutable Iceberg snapshot**. This gives us a full audit trail of the snapshot table's evolution — we can query any previous version of the data.

### Snapshot History

In [18]:
print("📸 Snapshot versions of users_snapshot:")
print()
run_query('SELECT * FROM iceberg.silver."users_snapshot$snapshots" ORDER BY committed_at')

📸 Snapshot versions of users_snapshot:

committed_at                     | snapshot_id         | parent_id           | operation | manifest_list                                                                                                                                       | summary                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             
---------------------------------+---------------------+---------------------+-----------+------------------------------------------------

[[datetime.datetime(2026, 2, 25, 16, 12, 34, 753000, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  2232473141858934015,
  None,
  'append',
  's3://lakehouse/silver/users_snapshot-d7f9bec677c94b4f82ee272edd4f0166/metadata/snap-2232473141858934015-1-95921a30-7a1d-4bbb-bb87-76a0cb600e26.avro',
  {'trino_query_id': '20260225_091637_00003_k4ixh',
   'trino_user': 'admin',
   'changed-partition-count': '0',
   'total-records': '0',
   'total-files-size': '0',
   'total-data-files': '0',
   'total-delete-files': '0',
   'total-position-deletes': '0',
   'total-equality-deletes': '0',
   'engine-version': '479',
   'engine-name': 'trino',
   'iceberg-version': 'Apache Iceberg 1.10.0 (commit 2114bf631e49af532d66e2ce148ee49dd1dd1f1f)'}],
 [datetime.datetime(2026, 2, 25, 16, 12, 42, 261000, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  7184175110500417767,
  2232473141858934015,
  'overwrite',
  's3://lakehouse/silver/users_snapshot-d7f9bec677c94b4f82ee272edd4f0166/metadata/snap-7184175110500417767-1-aefa8c

### Table History

In [19]:
print("📜 Full table history:")
print()
run_query('SELECT * FROM iceberg.silver."users_snapshot$history" ORDER BY made_current_at')

📜 Full table history:

made_current_at                  | snapshot_id         | parent_id           | is_current_ancestor
---------------------------------+---------------------+---------------------+--------------------
2026-02-25 16:12:34.753000+00:00 | 2232473141858934015 | None                | True               
2026-02-25 16:12:42.261000+00:00 | 7184175110500417767 | 2232473141858934015 | True               
2026-02-25 16:13:38.065000+00:00 | 6133278938024086751 | 7184175110500417767 | True               
2026-02-25 16:14:08.068000+00:00 | 4990089574103425397 | 6133278938024086751 | True               


[[datetime.datetime(2026, 2, 25, 16, 12, 34, 753000, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  2232473141858934015,
  None,
  True],
 [datetime.datetime(2026, 2, 25, 16, 12, 42, 261000, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  7184175110500417767,
  2232473141858934015,
  True],
 [datetime.datetime(2026, 2, 25, 16, 13, 38, 65000, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  6133278938024086751,
  7184175110500417767,
  True],
 [datetime.datetime(2026, 2, 25, 16, 14, 8, 68000, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  4990089574103425397,
  6133278938024086751,
  True]]

### CDC Log Snapshots (Bronze)

In [20]:
print("📸 Bronze CDC log — append history:")
print()
run_query('SELECT * FROM iceberg.bronze."cdc_users_log$snapshots" ORDER BY committed_at')

📸 Bronze CDC log — append history:

committed_at                     | snapshot_id         | parent_id           | operation | manifest_list                                                                                                                                      | summary                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                
---------------------------------+---------------------+---------------------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------

[[datetime.datetime(2026, 2, 25, 16, 12, 24, 867000, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  6108911319429947036,
  None,
  'append',
  's3://lakehouse/bronze/cdc_users_log-6377e2c293224192951464ef94c3b80e/metadata/snap-6108911319429947036-1-a94bd16c-3ba7-49f5-801d-37cf025f1818.avro',
  {'trino_query_id': '20260225_091626_00001_k4ixh',
   'trino_user': 'admin',
   'changed-partition-count': '0',
   'total-records': '0',
   'total-files-size': '0',
   'total-data-files': '0',
   'total-delete-files': '0',
   'total-position-deletes': '0',
   'total-equality-deletes': '0',
   'engine-version': '479',
   'engine-name': 'trino',
   'iceberg-version': 'Apache Iceberg 1.10.0 (commit 2114bf631e49af532d66e2ce148ee49dd1dd1f1f)'}],
 [datetime.datetime(2026, 2, 25, 16, 12, 27, 83000, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  8873339196143615553,
  6108911319429947036,
  'append',
  's3://lakehouse/bronze/cdc_users_log-6377e2c293224192951464ef94c3b80e/metadata/snap-8873339196143615553-1-41ee5e5a-cd2

---
## 📊 Summary

| Concept | What We Did |
|---------|------------|
| **Bronze Layer** | Append-only CDC ledger — every `I`/`U`/`D` event is an immutable row |
| **Deduplication** | `ROW_NUMBER()` window function isolates the latest event per `user_id` |
| **MERGE INTO** | Iceberg's upsert — handles INSERT, UPDATE, and DELETE in one statement |
| **Silver Layer** | Materialized current-state snapshot, always up-to-date |
| **Time Travel** | Iceberg snapshots provide full version history of every merge |

### 🏗️ Production Considerations

- **Incremental processing**: In production, you'd track a high-water mark (e.g., `event_ts > last_processed_ts`) to avoid re-processing the entire CDC log on every run
- **Partitioning**: Partition the Bronze CDC log by date to enable efficient time-range pruning
- **Compaction**: Periodically run Iceberg's `OPTIMIZE` and `expire_snapshots` to reclaim storage
- **Scheduling**: Use Airflow, Dagster, or a cron job to run the MERGE pipeline on a schedule
- **Schema evolution**: Iceberg supports schema evolution natively — new columns in CDC events are handled gracefully

---
## 🧹 Cleanup (Optional)

Drop all tables created by this notebook.

In [21]:
# Uncomment to drop all tables:
# run_query("DROP TABLE IF EXISTS iceberg.silver.users_snapshot")
# run_query("DROP TABLE IF EXISTS iceberg.bronze.cdc_users_log")
# print("🗑️ All CDC tables dropped")